In [1]:
from datasets import load_dataset

/lustre/fswork/projects/rech/tec/uod68bo/am/planktonzilla/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("project-oceania/planktonzilla_full")


Generating train split: 100%|██████████| 9686334/9686334 [02:14<00:00, 71985.36 examples/s] 


In [73]:
from dataclasses import dataclass
from functools import partial
from typing import Callable

import numpy as np
import torch
from datasets import Dataset, load_dataset
from transformers import AutoImageProcessor
from tqdm import tqdm

def compute_mean_and_std_dev(huggingface_dataset: Dataset):
    sum_pixels = np.zeros(3)  # For R, G, B channels
    sum_squared_pixels = np.zeros(3)
    num_pixels = 0

    for i in tqdm(range(len(huggingface_dataset))):
        # Access the image (assuming it's a PIL Image object)
        try:
            image = huggingface_dataset[i]["image"].convert("RGB")

        except Exception:
            continue

        # Convert image to NumPy array and normalize to [0, 1] if needed
        image_array = np.array(image).astype(np.float32) / 255.0

        # Reshape the image to (height * width, channels) to easily work with pixels
        if len(image_array.shape) == 3:
            # it is a color image with three channels
            reshaped_image = image_array.reshape(-1, 3)
        else:
            raise ValueError(f"Unsupported image_array shape: {image_array.shape}")

        # Accumulate sums
        sum_pixels += np.sum(reshaped_image, axis=0)
        sum_squared_pixels += np.sum(reshaped_image**2, axis=0)

        # Update total number of pixels
        num_pixels += reshaped_image.shape[0]

    mean = sum_pixels / num_pixels
    std_dev = np.sqrt((sum_squared_pixels / num_pixels) - (mean**2))


    return mean, std_dev


In [74]:
means, stds = compute_mean_and_std_dev(dataset["train"].take(100))

100%|██████████| 100/100 [00:00<00:00, 1252.31it/s]


In [76]:
stds

array([0.25828849, 0.25828849, 0.25828849])

In [7]:
means, stds = compute_mean_and_std_dev(dataset["train"])

  7%|▋         | 713793/9686334 [04:15<53:25, 2799.06it/s]  


UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x14b3e1011f80>